In [1]:
import pandas as pd
import os
import requests
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
import os
import requests

key = os.getenv("apikey")
pageSize = 50000
offset = 0
limit = 2000000
base = "https://health.data.ny.gov/resource/gnzp-ekau.json"

def get_data_from_api(base, offset, pageSize, key, limit):
    all_records = []
    num_of_calls = 0
    
    while len(all_records) < limit:
        # 1. Put all your query parameters into a dictionary
        query_params = {
            "$limit": pageSize,
            "$offset": offset,
            "$$app_token": key,
            "$where": "health_service_area='New York City'"
        }
        
        # 2. Pass the dictionary to the 'params' argument
        response = requests.get(base, params=query_params)
        
        if response.status_code == 200:
            data = response.json()
            
            if len(data) == 0:
                break
                
            all_records.extend(data)
            offset += pageSize
            num_of_calls += 1
            
            if len(data) < pageSize:
                break
        else:
            print("Data Load Unsuccessful!!")
            print(f"Error Code: {response.status_code}, {response.text}")
            break
            
    print(f"Number of rows loaded: {len(all_records)}")
    print(f"Number of API Calls: {num_of_calls}")
    
    # 3. Ensure we don't accidentally return more than the limit 
    # (e.g., if limit was 1200, we'd fetch 1500 without this slice)
    return all_records[:limit]

data = get_data_from_api(base, offset, pageSize, key, limit)

Number of rows loaded: 1074266
Number of API Calls: 22


In [13]:
db = pd.DataFrame(data)
db.head()

,health_service_area,hospital_county,operating_certificate_number,facility_id,facility_name,age_group,zip_code_3_digits,gender,race,ethnicity,...,apr_risk_of_mortality,apr_medical_surgical_description,payment_typology_1,birth_weight,abortion_edit_indicator,emergency_department_indicator,total_charges,total_costs,payment_typology_2,payment_typology_3
0,New York City,Bronx,7000001,1164,Bronx-Lebanon Hospital Center - Fulton Division,30 to 49,114,M,Other Race,Spanish/Hispanic,...,Minor,Medical,Medicaid,0,N,Y,8225.00,7341.60,NaN,NaN
1,New York City,Bronx,7000002,1165,Jacobi Medical Center,30 to 49,104,M,Black/African American,Not Span/Hispanic,...,Minor,Medical,Medicaid,0,N,Y,4846.76,2801.79,NaN,NaN
2,New York City,Bronx,7000002,1165,Jacobi Medical Center,0 to 17,104,M,Other Race,Spanish/Hispanic,...,Minor,Medical,Medicaid,0,N,Y,17205.30,9945.95,Medicaid,NaN
3,New York City,Bronx,7000002,1165,Jacobi Medical Center,30 to 49,104,F,Black/African American,Not Span/Hispanic,...,Minor,Medical,Medicaid,0,N,Y,9132.22,5279.11,NaN,NaN
4,New York City,Bronx,7000002,1165,Jacobi Medical Center,0 to 17,104,F,Other Race,Spanish/Hispanic,...,Minor,Medical,Medicaid,0,N,Y,6645.30,3841.48,NaN,NaN


In [14]:
print(db.columns)

Index(['health_service_area', 'hospital_county',
       'operating_certificate_number', 'facility_id', 'facility_name',
       'age_group', 'zip_code_3_digits', 'gender', 'race', 'ethnicity',
       'length_of_stay', 'type_of_admission', 'patient_disposition',
       'discharge_year', 'ccs_diagnosis_code', 'ccs_diagnosis_description',
       'ccs_procedure_code', 'ccs_procedure_description', 'apr_drg_code',
       'apr_drg_description', 'apr_mdc_code', 'apr_mdc_description',
       'apr_severity_of_illness_code', 'apr_severity_of_illness_description',
       'apr_risk_of_mortality', 'apr_medical_surgical_description',
       'payment_typology_1', 'birth_weight', 'abortion_edit_indicator',
       'emergency_department_indicator', 'total_charges', 'total_costs',
       'payment_typology_2', 'payment_typology_3'],
      dtype='str')


In [15]:
db.to_csv("data/main.csv", index=False)